# Phase 5 - Gap Analysis and Production Readiness

This notebook documents what the current indicator can and cannot claim, and which telemetry additions would close each remaining gap.


In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
ART = ROOT / "artifacts"
FIG = ROOT / "outputs" / "figures"

phase4_path = ART / "phase4_scores.csv"
phase4 = pd.read_csv(phase4_path) if phase4_path.exists() else None

print(f"Phase 4 score table found: {phase4_path.exists()}")
if phase4 is not None:
    print(f"Fixtures total={len(phase4)}; scored={(phase4['score_status']=='Scored').sum()}; suppressed={(phase4['score_status']!='Scored').sum()}")


Phase 4 score table found: True
Fixtures total=170; scored=16; suppressed=154


## 5.1 What The Indicator Cannot Do

- **No RUL estimate.** The indicator ranks relative degradation risk but does not model time-to-failure.
- **No lumen-depreciation calibration.** There is no direct photometric stream (lux/lumen proxy), so electrical stress is not directly tied to light output loss.
- **No definitive LED-versus-driver attribution.** `S_LED` and `S_driver` are physics-motivated partitions over shared electrical observables, not independently verified failure channels.
- **No full environmental isolation.** Ambient is not a reliable thermal anchor for junction temperature; humidity/pressure channels are unvalidated against failure outcomes.

### Section summary
The current output is a defensible **maintenance-priority indicator**, not a calibrated prognostics model.


## 5.2 Data Streams That Close Each Gap

1. **`inputCurrent` (Stream 1)**  
   Verifies CCR constant-current behavior over time and converts the central assumption into a measured condition.

2. **LED junction temperature telemetry**  
   Enables Arrhenius acceleration modeling and deconfounds thermal modulation from degradation drift.

3. **Photometric proxy (on-board light sensor)**  
   Links electrical stress to delivered light output and enables lumen-maintenance calibration.

4. **Runtime counter per fixture**  
   Converts calendar-time drift to usage-normalized degradation rate.

5. **Maintenance log with failure mode labels (LED vs driver)**  
   Supports supervised calibration and asymmetric weighting between sub-scores.

6. **RF / communication-health metrics**  
   Restores a true negative-control lane (communications-degraded but electrically healthy).

### Section summary
Each missing stream maps directly to one current claim limitation; the gap list is operational, not generic.


## 5.3 Centrepiece Gap: Missing Stream 1 (`inputCurrent`)

The primary leg relies on:

\[
V_{boost} = N \cdot V_f(I_{loop}, T_j) + V_{driver,drop}
\]

The model assumes loop current is held constant by CCR. Without `inputCurrent`, this cannot be verified in-data.

Implication:
- A rise in `boostVoltage` is interpreted as forward-voltage drift and/or driver stress.
- But if current setpoints drifted, part of the signal could be operational rather than aging.

Therefore, the indicator remains **defensible but assumption-bound**. Stream 1 is the single highest-value telemetry addition to move from assumption-based ranking to production-grade causal confidence.

### Section summary
Missing `inputCurrent` is the load-bearing uncertainty of the entire indicator.


## 5.4 Competition-Ready Positioning

- The method is honest about uncertainty and avoids overclaiming.
- The ranking is actionable for triage among fixtures with sufficient data.
- Suppression logic prevents low-quality fixtures from polluting primary decisions.
- Remaining gaps are explicit and directly connected to concrete telemetry asks.
